In [1]:
import os
from pathlib import Path

import gspread
import pandas as pd
from google.oauth2.service_account import Credentials
from merge_tables.db.connection import connect_to_postgres_via_duckdb

duck = connect_to_postgres_via_duckdb()

# From URL: https://docs.google.com/spreadsheets/d/<SPREADSHEET_ID>/edit
SPREADSHEET_ID = "1SBXeTGYQrmQXCCDQz21XGJ9iKE0HM-VZP8r69P5TOZ0"
SHEET_NAME = "1.Clientlist"

# credentials_path = os.environ.get("GOOGLE_APPLICATION_CREDENTIALS")
credentials_path = "./config/gsheet-creds.json"
# Or set explicitly: credentials_path = Path.home() / "secrets" / "service-account.json"
if not credentials_path:
    raise FileNotFoundError(
        "Set GOOGLE_APPLICATION_CREDENTIALS to your service account JSON path"
    )
credentials_path = Path(credentials_path).expanduser()


# Run the first gspread config cell so `credentials_path`, `SPREADSHEET_ID`, and `SHEET_NAME` exist.


def _sql_literal(s: str) -> str:
    return s.replace("'", "''")


key_path = _sql_literal(str(credentials_path.resolve()))

duck.execute("INSTALL gsheets FROM community;")
duck.execute("LOAD gsheets;")
duck.execute(
    f"""
CREATE OR REPLACE SECRET gsheet_sa (
    TYPE gsheet,
    PROVIDER key_file,
    FILEPATH '{key_path}'
);
"""
)

✓ Successfully connected DuckDB to PostgreSQL database 'medisoft'


In [2]:
from firms_join_table.spreadsheet_extract.config.mapping_spreadsheets import SPREADSHEET_MAPPING

In [3]:
DUSSELDORF = SPREADSHEET_MAPPING.sheets[0]
FRANKFURT = SPREADSHEET_MAPPING.sheets[1]
HAMBURG = SPREADSHEET_MAPPING.sheets[2]
KIEL = SPREADSHEET_MAPPING.sheets[3]
KOELN = SPREADSHEET_MAPPING.sheets[4]
MUENCHEN = SPREADSHEET_MAPPING.sheets[5]
ROSTOCK = SPREADSHEET_MAPPING.sheets[6]
STUTTGART = SPREADSHEET_MAPPING.sheets[7]
VIERSEN = SPREADSHEET_MAPPING.sheets[8]
BERLIN = SPREADSHEET_MAPPING.sheets[9]


In [6]:
SPREADSHEET_ID = '1vB84YG3eBVJVAQsv8VNe2K_TTE8bZ1I9gnwidGAvyXE'

In [4]:
# all_varchar=true avoids cast errors when Sheets mixes text/emoji/booleans in one column
duck.sql(
    f"""
SELECT * FROM read_gsheet(
    '{_sql_literal('1vB84YG3eBVJVAQsv8VNe2K_TTE8bZ1I9gnwidGAvyXE')}',
    sheet='{_sql_literal('berlin')}',
    all_varchar=true
)
"""
)

┌──────────────────────────────────────┬────────────────────────────────────────┬──────────────────────────────────────────┬───────────────────────────────────────────────────────────────────┬─────────────┬─────────────────────┬───────────────────────────────────────────┬─────────────────────────┬────────────────────┬──────────────┬─────────────────────┬──────────────┬─────────┬─────────┐
│             medisoft_id              │                  name                  │                 kuerzel                  │                               pfad                                │ nb_patients │   last_exam_date    │                 addresse                  │ has easybill connection │ link to connection │ no migration │ migrate as inactive │ Selbstzahler │  batch  │  todo   │
│               varchar                │                varchar                 │                 varchar                  │                              varchar                              │   varchar   │       var

In [25]:
duck.sql(
    f"""
    with rostock_dusseldorf as (
        select 
            rostock.easybill_kundennummer as easybill_id, 
            rostock.medisoft_ids as rostock_medisoft, 
            dusseldorf.medisoft_ids as dusseldorf_medisoft
        from read_gsheet(
            '{ROSTOCK.spreadsheet_id}',
            sheet='{ROSTOCK.easybill_spreadsheet_name}',
            all_varchar=true
            ) as rostock
        left join read_gsheet('{DUSSELDORF.spreadsheet_id}',
            sheet='{DUSSELDORF.easybill_spreadsheet_name}',
            all_varchar=true
            ) as dusseldorf
            on dusseldorf.easybill_kundennummer = rostock.easybill_kundennummer
    )
    select * from rostock_dusseldorf where rostock_medisoft != dusseldorf_medisoft
    """
)

┌─────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│ easybill_id │                                                                                                                               

## Merge all city clientlists → one table

**Rostock** is the base row set (most recent & complete — a strict superset of every other city's clients). For each client, `medisoft_ids` becomes the **union** of that client's medisoft matches across **all** city sheets: split on newlines, **deduplicated**, and re-joined with newlines. Every other column is taken from Rostock.

Output: `output/rostock_merged_clientlist.csv`.

In [49]:
# Pull every city's "1.Clientlist" tab live and stack them (one row per client per city),
# keeping all columns + a `city` tag. UNION ALL BY NAME tolerates differing column orders.
def _all_clientlists_union_sql() -> str:
    blocks = [
        f"""
        SELECT *, '{_sql_literal(s.city)}' AS city
        FROM read_gsheet(
            '{_sql_literal(s.spreadsheet_id)}',
            sheet='{_sql_literal(s.easybill_spreadsheet_name)}',
            all_varchar=true
        )"""
        for s in SPREADSHEET_MAPPING.sheets
    ]
    return " UNION ALL BY NAME ".join(blocks)


duck.execute(
    f"CREATE OR REPLACE TABLE all_city_clientlists AS {_all_clientlists_union_sql()}"
)
duck.sql("SELECT city, count(*) AS n_clients FROM all_city_clientlists GROUP BY city ORDER BY city")

KeyboardInterrupt: 

In [50]:
duck.execute(
    """
    create or replace table rostock_merged_clientlist as 
    select * from read_csv('/Users/adrienblanquer/code/bas-utils/firms_join_table/spreadsheet_extract/output/rostock_merged_clientlist.csv')
    """
)

In [ ]:
# Rostock = base rows (most recent & complete superset of every other city's clients).
# medisoft_ids = deduped union of every city's matches per client: split on newlines,
# distinct, re-joined with newlines (sorted for determinism). Other columns come from Rostock.
duck.execute(
    """
    CREATE OR REPLACE TABLE rostock_merged_clientlist AS
    WITH tokens AS (
        SELECT * FROM (
            SELECT
                easybill_kundennummer,
                trim(unnest(string_split(trim(medisoft_ids), chr(10)))) AS medisoft_id
            FROM all_city_clientlists
            WHERE medisoft_ids IS NOT NULL AND trim(medisoft_ids) <> ''
        ) WHERE medisoft_id <> ''
    ),
    merged_medisoft AS (
        SELECT
            easybill_kundennummer,
            string_agg(medisoft_id, chr(10) ORDER BY medisoft_id) AS medisoft_ids
        FROM (SELECT DISTINCT easybill_kundennummer, medisoft_id FROM tokens)
        GROUP BY easybill_kundennummer
    )
    SELECT r.* EXCLUDE (city) REPLACE (m.medisoft_ids AS medisoft_ids)
    FROM all_city_clientlists r
    LEFT JOIN merged_medisoft m USING (easybill_kundennummer)
    WHERE r.city = 'Rostock'
    """
)
duck.sql("SELECT * FROM rostock_merged_clientlist ORDER BY easybill_kundennummer")

┌───────────────────────┬───────────────────┬──────────────┬───────────┬─────────────────┬─────────────────────────────────────────────────────────────────┬───────────────┬──────────────────┬───────────────────────────────────────┬────────────────────────────────────────────────────────────────────────────┬───────────────────────────────────────────────────────┬────────────────────┬─────────────────────────┬─────────────────────────┬───────────┬──────────────┐
│ easybill_kundennummer │ last_invoice_date │ € net billed │ spe. care │ wochenliste_ids │                         easybill_firma                          │ easybill_name │ easybill_vorname │           easybill_address            │                                medisoft_ids                                │                    medisoft_names                     │     sim_scores     │         zoho_id         │      link to zoho       │ validated │ no_migration │
│        varchar        │      varchar      │   varchar    │  varchar 

In [ ]:
# Coverage gained by merging, plus a guard that no cell ended up with duplicate medisoft ids.
stats = duck.sql(
    """
    WITH base AS (SELECT * FROM all_city_clientlists WHERE city = 'Rostock'),
         merged AS (SELECT * FROM rostock_merged_clientlist)
    SELECT
        (SELECT count(*) FROM merged)                                              AS clients,
        (SELECT count(*) FROM base   WHERE trim(coalesce(medisoft_ids, '')) <> '') AS matched_rostock_only,
        (SELECT count(*) FROM merged WHERE trim(coalesce(medisoft_ids, '')) <> '') AS matched_after_merge,
        (SELECT count(*) FROM (
            SELECT unnest(string_split(medisoft_ids, chr(10)))
            FROM merged WHERE medisoft_ids IS NOT NULL
        ))                                                                         AS total_client_medisoft_pairs
    """
).df()

dup_rows = duck.sql(
    """
    WITH s AS (
        SELECT string_split(medisoft_ids, chr(10)) AS toks
        FROM rostock_merged_clientlist
        WHERE medisoft_ids IS NOT NULL
    )
    SELECT count(*) AS n FROM s WHERE len(toks) <> len(list_distinct(toks))
    """
).fetchone()[0]
assert dup_rows == 0, f"{dup_rows} cells still contain duplicate medisoft ids"
print("no duplicate medisoft ids within any cell ✓")
stats

no duplicate medisoft ids within any cell ✓


,clients,matched_rostock_only,matched_after_merge,total_client_medisoft_pairs
0,3536,1411,1760,2515


In [68]:
from pathlib import Path

merged_csv = Path("output") / "rostock_merged_clientlist_2.csv"
merged_csv.parent.mkdir(parents=True, exist_ok=True)
duck.execute(
    f"COPY rostock_merged_clientlist_2 TO '{_sql_literal(str(merged_csv.resolve()))}' (HEADER, DELIMITER ',')"
)
print(f"Wrote {merged_csv.resolve()}")

Wrote /Users/adrienblanquer/code/bas-utils/firms_join_table/spreadsheet_extract/output/rostock_merged_clientlist_2.csv


In [65]:
duck.sql(
    """
    with easybill_med as (
        select easybill_kundennummer, unnest(split(medisoft_ids, '\n'))  as medisoft_id
        from rostock_merged_clientlist
    )
    select easybill_kundennummer, medisoft_id, f.pfad as pfad
    from easybill_med r
    left join pg.medisoft.table_firmenstruktur_save f
        on r.medisoft_id = f.rec_id
    """
)

┌───────────────────────┬───────────────┬──────────────────────────────────────────────────────────────┐
│ easybill_kundennummer │  medisoft_id  │                             pfad                             │
│         int64         │    varchar    │                           varchar                            │
├───────────────────────┼───────────────┼──────────────────────────────────────────────────────────────┤
│             130002124 │ 00_8JJ00VTGCE │ Change                                                       │
│             130002124 │ 00_A1W00VHI68 │ BSH Düsseldorf / .change GmbH                                │
│             130000200 │ 00_98L00R34FI │ BSH Rostock / Auf der Tenne Dummerstorf                      │
│             130000200 │ 00_9F600U7MB1 │ BSH Rostock / „Auf der Tenne“ e.V.löschen!                   │
│             130002066 │ 00_A0S00RIHE7 │ BSH Rostock / Die 3 - Transport- und Handelsgesellschaft mbH │
│             111000009 │ 00_8PG00L8MCI │ BSH Hamburg  

In [67]:
# Rebuild rostock_merged_clientlist with `medisoft_names` replaced by the firm path (pfad),
# looked up per id and kept aligned 1:1 with `medisoft_ids`.
#  - trim() on each unnested id clears stray spaces so the rec_id join matches (and cells stay clean)
#  - SAME `ORDER BY medisoft_id` in both string_agg keeps ids and names on matching lines
#  - coalesce(pfad,'?') stops ids missing from firmenstruktur from dropping a name and shifting alignment
# Rebuilt in place and idempotent (re-running is safe).
duck.execute(
    """
    CREATE OR REPLACE TABLE rostock_merged_clientlist_2 AS
    WITH easybill_med AS (
        SELECT
            easybill_kundennummer,
            trim(unnest(split(medisoft_ids, chr(10)))) AS medisoft_id
        FROM rostock_merged_clientlist
        WHERE medisoft_ids IS NOT NULL
    ),
    enriched AS (
        SELECT r.easybill_kundennummer, r.medisoft_id, f.pfad
        FROM easybill_med r
        LEFT JOIN pg.medisoft.table_firmenstruktur_save f
            ON r.medisoft_id = f.rec_id
        WHERE r.medisoft_id <> ''
    ),
    agg AS (
        SELECT
            easybill_kundennummer,
            string_agg(medisoft_id, chr(10) ORDER BY medisoft_id)         AS medisoft_ids,
            string_agg(coalesce(pfad, '?'), chr(10) ORDER BY medisoft_id) AS medisoft_names
        FROM enriched
        GROUP BY easybill_kundennummer
    )
    SELECT base.* REPLACE (
        agg.medisoft_ids   AS medisoft_ids,
        agg.medisoft_names AS medisoft_names
    )
    FROM rostock_merged_clientlist base
    LEFT JOIN agg USING (easybill_kundennummer)
    """
)
duck.sql(
    """
    SELECT easybill_kundennummer, medisoft_ids, medisoft_names
    FROM rostock_merged_clientlist_2
    WHERE medisoft_ids IS NOT NULL
    ORDER BY easybill_kundennummer
    """
)

┌───────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│ easybill_kundennummer │                                                             medisoft_ids                                                              │                                                                                                                                                                                                                    medisoft_names           

In [33]:
DEST_SHEET_ID = '1vB84YG3eBVJVAQsv8VNe2K_TTE8bZ1I9gnwidGAvyXE'
easybill_sheet = 'easybill_clients_list'

In [71]:
duck.sql("select * from pg.bas_firms.easybill_medisoft")

┌───────────┬─────────────┬──────────────────────────────────────┐
│    id     │ easybill_id │             medisoft_id              │
│  varchar  │   varchar   │               varchar                │
├───────────┼─────────────┼──────────────────────────────────────┤
│ 100020025 │ 100020025   │ NULL                                 │
│ 100020026 │ 100020026   │ NULL                                 │
│ 100020027 │ 100020027   │ NULL                                 │
│ 100020028 │ 100020028   │ 00_8M000MVTNG                        │
│ 10073     │ 10073       │ NULL                                 │
│ 104000004 │ 104000004   │ 8D3F647A-94B6-4F9D-874D-EA1AEFB43C07 │
│ 104000005 │ 104000005   │ CE1ACAF4-BE06-47AD-B29D-67E4294C82F0 │
│ 104000007 │ 104000007   │ NULL                                 │
│ 104000010 │ 104000010   │ NULL                                 │
│ 104000011 │ 104000011   │ NULL                                 │
│     ·     │  ·          │  ·                                

In [5]:
duck.sql(
    """
    select distinct lower(mandant) from pg.medisoft.table_firmenstruktur_save order by 1
    """
)

┌────────────────┐
│ lower(mandant) │
│    varchar     │
├────────────────┤
│ berlin         │
│ change         │
│ düsseldorf     │
│ frankfurt      │
│ hamburg        │
│ kiel           │
│ köln           │
│ münchen        │
│ rostock        │
│ stuttgart      │
│ team-energie   │
│ viersen        │
│ NULL           │
├────────────────┤
│    13 rows     │
└────────────────┘

In [34]:
duck.sql(
        f"""
    select 
        f.rec_id as medisoft_id,
        name,
        kuerzel,
        pfad,
        count(distinct b.rec_id) as nb_patients,
        max(u.u_datum) as last_exam_date, 
        concat_ws(' ', f.strasse, f.plz, f.ort) as address
    from pg.medisoft.table_firmenstruktur_save  f
    left join pg.medisoft.table_beschaeftigte b
        on b.ebetrieb_id = f.rec_id
    left join pg.medisoft.table_untersuchungen u
        on f.rec_id = u.abetrieb_id or u.besch_id = b.rec_id
    where lower(mandant) is null
    group by 1,2,3,4,7
    """
    #select pfad, * 
    # from pg.medisoft.table_firmenstruktur_save where lower(mandant) in ('change', 'team-energie', NULL)
)
#.to_csv('update_june_other.csv')

┌──────────────────────────────────────┬────────────────────────────────────┬────────────────────────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────┬─────────────┬─────────────────────┬───────────────────────────────────────────┐
│             medisoft_id              │                name                │              kuerzel               │                                            pfad                                             │ nb_patients │   last_exam_date    │                  address                  │
│               varchar                │              varchar               │              varchar               │                                           varchar                                           │    int64    │       varchar       │                  varchar                  │
├──────────────────────────────────────┼────────────────────────────────────┼────────────────────────────────────┼───────────────────

In [9]:
# berlin
duck.sql(
    f"""
    select 
        f.rec_id as medisoft_id,
        name,
        kuerzel,
        pfad,
        count(distinct b.rec_id) as nb_patients,
        max(u.u_datum) as last_exam_date, 
        concat_ws(' ', f.strasse, f.plz, f.ort) as address
    from pg.medisoft.table_firmenstruktur_save  f
    left join pg.medisoft.table_beschaeftigte b
        on b.ebetrieb_id = f.rec_id
    left join pg.medisoft.table_untersuchungen u
        on f.rec_id = u.abetrieb_id or u.besch_id = b.rec_id
    where lower(pfad) like 'bsh berlin%' and f.rec_id not in (
        select medisoft_id from read_gsheet(
            '{SPREADSHEET_ID}',
            sheet='berlin',
            all_varchar=true
            )
        ) 
    group by 1,2,3,4,7
    """
).to_csv('update_july_berlin.csv')

In [ ]:
#stuttgart
duck.sql(
    f"""
    select 
        f.rec_id as medisoft_id,
        name,
        kuerzel,
        pfad,
        count(distinct b.rec_id) as nb_patients,
        max(u.u_datum) as last_exam_date, 
        concat_ws(' ', f.strasse, f.plz, f.ort) as address
    from pg.medisoft.table_firmenstruktur_save  f
    left join pg.medisoft.table_beschaeftigte b
        on b.ebetrieb_id = f.rec_id
    left join pg.medisoft.table_untersuchungen u
        on f.rec_id = u.abetrieb_id or u.besch_id = b.rec_id
    where lower(pfad) like 'bsh stuttgart%' and f.rec_id not in (
        select medisoft_id from read_gsheet(
            '{SPREADSHEET_ID}',
            sheet='stuttgart',
            all_varchar=true
            )
        ) 
    group by 1,2,3,4,7
    """
).to_csv('update_july_stuttgart.csv')

In [31]:
#munchen
duck.sql(
    f"""
    select 
        f.rec_id as medisoft_id,
        name,
        kuerzel,
        pfad,
        count(distinct b.rec_id) as nb_patients,
        max(u.u_datum) as last_exam_date, 
        concat_ws(' ', f.strasse, f.plz, f.ort) as address,
        '=RECHERCHEX("*" & A2 & "*"; easybill_clients_list!$J$2:J; easybill_clients_list!$J$2:J; "Not found"; 2) <> "Not found"' as has_easybill_connection,
        '=SI(A2<>""; LET(row_index; EQUIV("*" & SUPPRESPACE(A2) & "*"; easybill_clients_list!$J:$J; 0); SI(ESTERREUR(row_index); "Not found"; LIEN_HYPERTEXTE("#gid=2062502986&range=I" & row_index; "👉 Check the cell"))); "")' as link_to_connection,
        False as migrate_as_inactive,
        False as no_migration,
        False as Selbstzahler,
        'june' as batch,
        '=H2+J2+K2=0' as todo
    from pg.medisoft.table_firmenstruktur_save  f
    left join pg.medisoft.table_beschaeftigte b
        on b.ebetrieb_id = f.rec_id
    left join pg.medisoft.table_untersuchungen u
        on f.rec_id = u.abetrieb_id or u.besch_id = b.rec_id
    where lower(pfad) like '%bsh münchen%' and f.rec_id not in (
        select medisoft_id from read_gsheet(
            '{SPREADSHEET_ID}',
            sheet='munchen',
            all_varchar=true
            )
        ) 
    group by 1,2,3,4,7
    """
)
#.to_csv('update_june_munchen.csv')

┌─────────────┬─────────┬─────────┬─────────┬─────────────┬────────────────┬─────────┬─────────────────────────┬────────────────────┬─────────────────────┬──────────────┬──────────────┬─────────┬─────────┐
│ medisoft_id │  name   │ kuerzel │  pfad   │ nb_patients │ last_exam_date │ address │ has_easybill_connection │ link_to_connection │ migrate_as_inactive │ no_migration │ Selbstzahler │  batch  │  todo   │
│   varchar   │ varchar │ varchar │ varchar │    int64    │    varchar     │ varchar │         varchar         │      varchar       │       boolean       │   boolean    │   boolean    │ varchar │ varchar │
├─────────────┴─────────┴─────────┴─────────┴─────────────┴────────────────┴─────────┴─────────────────────────┴────────────────────┴─────────────────────┴──────────────┴──────────────┴─────────┴─────────┤
│                                                                                                  0 rows                                                                       

In [30]:
# köln
duck.sql(
    f"""
    select 
        f.rec_id as medisoft_id,
        name,
        kuerzel,
        pfad,
        count(distinct b.rec_id) as nb_patients,
        max(u.u_datum) as last_exam_date, 
        concat_ws(' ', f.strasse, f.plz, f.ort) as address,
        '=RECHERCHEX("*" & A2 & "*"; easybill_clients_list!$J$2:J; easybill_clients_list!$J$2:J; "Not found"; 2) <> "Not found"' as has_easybill_connection,
        '=SI(A2<>""; LET(row_index; EQUIV("*" & SUPPRESPACE(A2) & "*"; easybill_clients_list!$J:$J; 0); SI(ESTERREUR(row_index); "Not found"; LIEN_HYPERTEXTE("#gid=2062502986&range=I" & row_index; "👉 Check the cell"))); "")' as link_to_connection,
        False as migrate_as_inactive,
        False as no_migration,
        False as Selbstzahler,
        'june' as batch,
        '=H2+J2+K2=0' as todo
    from pg.medisoft.table_firmenstruktur_save  f
    left join pg.medisoft.table_beschaeftigte b
        on b.ebetrieb_id = f.rec_id
    left join pg.medisoft.table_untersuchungen u
        on f.rec_id = u.abetrieb_id or u.besch_id = b.rec_id
    where lower(pfad) like '%bsh köln%' and f.rec_id not in (
        select medisoft_id from read_gsheet(
            '{SPREADSHEET_ID}',
            sheet='koln',
            all_varchar=true
            )
        ) 
    group by 1,2,3,4,7
    """
).to_csv('update_july_koln.csv')

In [28]:
#kiel
duck.sql(
    f"""
    select 
        f.rec_id as medisoft_id,
        name,
        kuerzel,
        pfad,
        count(distinct b.rec_id) as nb_patients,
        max(u.u_datum) as last_exam_date, 
        concat_ws(' ', f.strasse, f.plz, f.ort) as address,
        '=RECHERCHEX("*" & A2 & "*"; easybill_clients_list!$J$2:J; easybill_clients_list!$J$2:J; "Not found"; 2) <> "Not found"' as has_easybill_connection,
        '=SI(A2<>""; LET(row_index; EQUIV("*" & SUPPRESPACE(A2) & "*"; easybill_clients_list!$J:$J; 0); SI(ESTERREUR(row_index); "Not found"; LIEN_HYPERTEXTE("#gid=2062502986&range=I" & row_index; "👉 Check the cell"))); "")' as link_to_connection,
        False as migrate_as_inactive,
        False as no_migration,
        False as Selbstzahler,
        'june' as batch,
        '=H2+J2+K2=0' as todo
    from pg.medisoft.table_firmenstruktur_save  f
    left join pg.medisoft.table_beschaeftigte b
        on b.ebetrieb_id = f.rec_id
    left join pg.medisoft.table_untersuchungen u
        on f.rec_id = u.abetrieb_id or u.besch_id = b.rec_id
    where lower(pfad) like 'bsh kiel%' and f.rec_id not in (
        select medisoft_id from read_gsheet(
            '{SPREADSHEET_ID}',
            sheet='kiel',
            all_varchar=true
            )
        ) 
    group by 1,2,3,4,7
    """
)

┌─────────────┬─────────┬─────────┬─────────┬─────────────┬────────────────┬─────────┬─────────────────────────┬────────────────────┬─────────────────────┬──────────────┬──────────────┬─────────┬─────────┐
│ medisoft_id │  name   │ kuerzel │  pfad   │ nb_patients │ last_exam_date │ address │ has_easybill_connection │ link_to_connection │ migrate_as_inactive │ no_migration │ Selbstzahler │  batch  │  todo   │
│   varchar   │ varchar │ varchar │ varchar │    int64    │    varchar     │ varchar │         varchar         │      varchar       │       boolean       │   boolean    │   boolean    │ varchar │ varchar │
├─────────────┴─────────┴─────────┴─────────┴─────────────┴────────────────┴─────────┴─────────────────────────┴────────────────────┴─────────────────────┴──────────────┴──────────────┴─────────┴─────────┤
│                                                                                                  0 rows                                                                       

In [27]:
#hamburg
duck.sql(
    f"""
    select 
        f.rec_id as medisoft_id,
        name,
        kuerzel,
        pfad,
        count(distinct b.rec_id) as nb_patients,
        max(u.u_datum) as last_exam_date, 
        concat_ws(' ', f.strasse, f.plz, f.ort) as address,
        '=RECHERCHEX("*" & A2 & "*"; easybill_clients_list!$J$2:J; easybill_clients_list!$J$2:J; "Not found"; 2) <> "Not found"' as has_easybill_connection,
        '=SI(A2<>""; LET(row_index; EQUIV("*" & SUPPRESPACE(A2) & "*"; easybill_clients_list!$J:$J; 0); SI(ESTERREUR(row_index); "Not found"; LIEN_HYPERTEXTE("#gid=2062502986&range=I" & row_index; "👉 Check the cell"))); "")' as link_to_connection,
        False as migrate_as_inactive,
        False as no_migration,
        False as Selbstzahler,
        'june' as batch,
        '=H2+J2+K2=0' as todo
    from pg.medisoft.table_firmenstruktur_save  f
    left join pg.medisoft.table_beschaeftigte b
        on b.ebetrieb_id = f.rec_id
    left join pg.medisoft.table_untersuchungen u
        on f.rec_id = u.abetrieb_id or u.besch_id = b.rec_id
    where lower(pfad) like '%bsh hamburg%' and f.rec_id not in (
        select medisoft_id from read_gsheet(
            '{SPREADSHEET_ID}',
            sheet='hamburg',
            all_varchar=true
            )
        ) 
    group by 1,2,3,4,7
    """
).to_csv('update_july_hamburg.csv')

In [14]:
#frankfurt
duck.sql(
    f"""
    select 
        f.rec_id as medisoft_id,
        name,
        kuerzel,
        pfad,
        count(distinct b.rec_id) as nb_patients,
        max(u.u_datum) as last_exam_date, 
        concat_ws(' ', f.strasse, f.plz, f.ort) as address,
        '=RECHERCHEX("*" & A2 & "*"; easybill_clients_list!$J$2:J; easybill_clients_list!$J$2:J; "Not found"; 2) <> "Not found"' as has_easybill_connection,
        '=SI(A2<>""; LET(row_index; EQUIV("*" & SUPPRESPACE(A2) & "*"; easybill_clients_list!$J:$J; 0); SI(ESTERREUR(row_index); "Not found"; LIEN_HYPERTEXTE("#gid=2062502986&range=I" & row_index; "👉 Check the cell"))); "")' as link_to_connection,
        False as migrate_as_inactive,
        False as no_migration,
        False as Selbstzahler,
        'june' as batch,
        '=H2+J2+K2=0' as todo
    from pg.medisoft.table_firmenstruktur_save  f
    left join pg.medisoft.table_beschaeftigte b
        on b.ebetrieb_id = f.rec_id
    left join pg.medisoft.table_untersuchungen u
        on f.rec_id = u.abetrieb_id or u.besch_id = b.rec_id
    where lower(pfad) like 'bsh frankfurt%' and f.rec_id not in (
        select medisoft_id from read_gsheet(
            '{SPREADSHEET_ID}',
            sheet='frankfurt',
            all_varchar=true
            )
        ) 
    group by 1,2,3,4,7
    """
).to_csv('update_july_frankfurt.csv')

In [10]:
#dusseldorf
duck.sql(
    f"""
    select 
        f.rec_id as medisoft_id,
        name,
        kuerzel,
        pfad,
        count(distinct b.rec_id) as nb_patients,
        max(u.u_datum) as last_exam_date, 
        concat_ws(' ', f.strasse, f.plz, f.ort) as address,
        '=RECHERCHEX("*" & A2 & "*"; easybill_clients_list!$J$2:J; easybill_clients_list!$J$2:J; "Not found"; 2) <> "Not found"' as has_easybill_connection,
        '=SI(A2<>""; LET(row_index; EQUIV("*" & SUPPRESPACE(A2) & "*"; easybill_clients_list!$J:$J; 0); SI(ESTERREUR(row_index); "Not found"; LIEN_HYPERTEXTE("#gid=2062502986&range=I" & row_index; "👉 Check the cell"))); "")' as link_to_connection,
        False as migrate_as_inactive,
        False as no_migration,
        False as Selbstzahler,
        'june' as batch,
        '=H2+J2+K2=0' as todo
    from pg.medisoft.table_firmenstruktur_save  f
    left join pg.medisoft.table_beschaeftigte b
        on b.ebetrieb_id = f.rec_id
    left join pg.medisoft.table_untersuchungen u
        on f.rec_id = u.abetrieb_id or u.besch_id = b.rec_id
    where lower(pfad) like 'bsh düsseldorf%' and f.rec_id not in (
        select medisoft_id from read_gsheet(
            '{SPREADSHEET_ID}',
            sheet='dusseldorf',
            all_varchar=true
            )
        ) 
    group by 1,2,3,4,7
    """
)
#.to_csv('update_june_dusseldorf.csv')

┌─────────────┬─────────┬─────────┬─────────┬─────────────┬────────────────┬─────────┬─────────────────────────┬────────────────────┬─────────────────────┬──────────────┬──────────────┬─────────┬─────────┐
│ medisoft_id │  name   │ kuerzel │  pfad   │ nb_patients │ last_exam_date │ address │ has_easybill_connection │ link_to_connection │ migrate_as_inactive │ no_migration │ Selbstzahler │  batch  │  todo   │
│   varchar   │ varchar │ varchar │ varchar │    int64    │    varchar     │ varchar │         varchar         │      varchar       │       boolean       │   boolean    │   boolean    │ varchar │ varchar │
├─────────────┴─────────┴─────────┴─────────┴─────────────┴────────────────┴─────────┴─────────────────────────┴────────────────────┴─────────────────────┴──────────────┴──────────────┴─────────┴─────────┤
│                                                                                                  0 rows                                                                       

In [12]:
#berlin
duck.sql(
    f"""
    select 
        f.rec_id as medisoft_id,
        name,
        kuerzel,
        pfad,
        count(distinct b.rec_id) as nb_patients,
        max(u.u_datum) as last_exam_date, 
        concat_ws(' ', f.strasse, f.plz, f.ort) as address,
        '=RECHERCHEX("*" & A2 & "*"; easybill_clients_list!$J$2:J; easybill_clients_list!$J$2:J; "Not found"; 2) <> "Not found"' as has_easybill_connection,
        '=SI(A2<>""; LET(row_index; EQUIV("*" & SUPPRESPACE(A2) & "*"; easybill_clients_list!$J:$J; 0); SI(ESTERREUR(row_index); "Not found"; LIEN_HYPERTEXTE("#gid=2062502986&range=I" & row_index; "👉 Check the cell"))); "")' as link_to_connection,
        False as migrate_as_inactive,
        False as no_migration,
        False as Selbstzahler,
        'june' as batch,
        '=H2+J2+K2=0' as todo
    from pg.medisoft.table_firmenstruktur_save  f
    left join pg.medisoft.table_beschaeftigte b
        on b.ebetrieb_id = f.rec_id
    left join pg.medisoft.table_untersuchungen u
        on f.rec_id = u.abetrieb_id or u.besch_id = b.rec_id
    where lower(pfad) like 'bsh berlin%' and f.rec_id not in (
        select medisoft_id from read_gsheet(
            '{SPREADSHEET_ID}',
            sheet='berlin',
            all_varchar=true
            )
        ) 
    group by 1,2,3,4,7
    """
)
#.to_csv('update_june_berlin.csv')

┌─────────────┬─────────┬─────────┬─────────┬─────────────┬────────────────┬─────────┬─────────────────────────┬────────────────────┬─────────────────────┬──────────────┬──────────────┬─────────┬─────────┐
│ medisoft_id │  name   │ kuerzel │  pfad   │ nb_patients │ last_exam_date │ address │ has_easybill_connection │ link_to_connection │ migrate_as_inactive │ no_migration │ Selbstzahler │  batch  │  todo   │
│   varchar   │ varchar │ varchar │ varchar │    int64    │    varchar     │ varchar │         varchar         │      varchar       │       boolean       │   boolean    │   boolean    │ varchar │ varchar │
├─────────────┴─────────┴─────────┴─────────┴─────────────┴────────────────┴─────────┴─────────────────────────┴────────────────────┴─────────────────────┴──────────────┴──────────────┴─────────┴─────────┤
│                                                                                                  0 rows                                                                       